# 06 — Explainability (SHAP Analysis)
## Global and Local Explanations for the Federated Ensemble

This notebook generates SHAP-based explanations for the trained models.
We use the BAF dataset for primary SHAP analysis since it has interpretable features
(income, age, credit_risk_score, velocity metrics etc.).

Produces: V19 (SHAP summary), V20 (SHAP bar), V21 (SHAP force),
          V22 (SHAP dependence), V23 (SHAP waterfall)

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.utils.parallel')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from pathlib import Path
import joblib

from src.explainability.shap_analysis import generate_shap_explanations

FIGURES_DIR = Path('../outputs/figures')
MODELS_DIR = Path('../outputs/models')
PROCESSED_DIR = Path('../data/processed')

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Model and Data

In [ ]:
# Load BAF test data (interpretable features)
X_test = pd.read_csv(PROCESSED_DIR / 'baf_X_test.csv')
y_test = pd.read_csv(PROCESSED_DIR / 'baf_y_test.csv').squeeze()

print(f'Test set: {X_test.shape[0]:,} samples, {X_test.shape[1]} features')
print(f'Fraud count in test: {y_test.sum():,} ({y_test.mean()*100:.2f}%)')
print(f'\nFeatures: {list(X_test.columns)}')

In [ ]:
# Load XGBoost model trained on BAF data (must match BAF test features)
# Try FL BAF-specific model first, then GA-optimized, then baseline
model_load_order = [
    (MODELS_DIR / 'fl_baf_xgb.joblib', 'FL BAF XGBoost'),
    (MODELS_DIR / 'ga_optimized_xgb.joblib', 'GA-Optimized XGBoost'),
    (MODELS_DIR / 'xgboost_baf.joblib', 'Centralized BAF XGBoost'),
    (MODELS_DIR / 'xgboost.joblib', 'Centralized XGBoost'),
]

xgb_model = None
for model_path, label in model_load_order:
    if model_path.exists():
        xgb_model = joblib.load(model_path)
        model_label = label
        # Verify feature count matches
        n_model_features = xgb_model.n_features_in_
        if n_model_features == X_test.shape[1]:
            print(f'Loaded: {label} ({n_model_features} features — matches test data)')
            break
        else:
            print(f'Skipped: {label} ({n_model_features} features != {X_test.shape[1]} test features)')
            xgb_model = None

if xgb_model is None:
    raise FileNotFoundError(
        f'No saved XGBoost model found with {X_test.shape[1]} features (BAF). '
        'Please run notebooks 03-05 first.'
    )

## 2. Generate SHAP Explanations (Automated Suite)

In [ ]:
print(f'Generating SHAP explanations for {model_label}...')
shap_values = generate_shap_explanations(
    model=xgb_model,
    X_test=X_test,
    y_test=y_test,
    feature_names=list(X_test.columns),
    output_dir=str(FIGURES_DIR),
    model_name='xgboost_global',
)
print('\nAll SHAP plots generated.')

## 3. Additional SHAP Analysis — Random Forest

In [ ]:
# Also explain RF for comparison — must match BAF features
rf_load_order = [
    (MODELS_DIR / 'fl_baf_rf.joblib', 'FL BAF Random Forest'),
    (MODELS_DIR / 'random_forest_baf.joblib', 'Centralized BAF RF'),
    (MODELS_DIR / 'random_forest.joblib', 'Centralized RF'),
]

rf_model = None
for rf_path, rf_label in rf_load_order:
    if rf_path.exists():
        rf_candidate = joblib.load(rf_path)
        if rf_candidate.n_features_in_ == X_test.shape[1]:
            rf_model = rf_candidate
            print(f'Loaded: {rf_label} ({rf_candidate.n_features_in_} features)')
            break
        else:
            print(f'Skipped: {rf_label} ({rf_candidate.n_features_in_} features != {X_test.shape[1]})')

if rf_model is not None:
    print(f'Generating SHAP explanations for {rf_label}...')
    shap_values_rf = generate_shap_explanations(
        model=rf_model,
        X_test=X_test,
        y_test=y_test,
        feature_names=list(X_test.columns),
        output_dir=str(FIGURES_DIR),
        model_name='rf_global',
    )
    print('RF SHAP plots generated.')
else:
    print('No RF model with matching features found, skipping.')

## 4. Feature Importance Comparison (XGB vs RF)

In [ ]:
# Compare top features between XGB and RF
xgb_importance = np.abs(shap_values).mean(axis=0)
top_k = 10

feature_names = list(X_test.columns)
sorted_idx = np.argsort(xgb_importance)[-top_k:]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    [feature_names[i] for i in sorted_idx],
    xgb_importance[sorted_idx],
    color='#3498db', edgecolor='black', linewidth=0.3,
)
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title(f'Top {top_k} Most Important Features ({model_label})', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_top_features_comparison.png', dpi=300)
plt.show()
print('Saved: shap_top_features_comparison.png')

## 5. Fraud vs Legitimate SHAP Distribution

In [ ]:
# Compare SHAP value distributions for fraud vs legitimate
# Use the subsample used in SHAP computation
max_samples = 1000
if len(X_test) > max_samples:
    sample_idx = np.random.RandomState(42).choice(len(X_test), max_samples, replace=False)
    y_explain = y_test.iloc[sample_idx]
else:
    y_explain = y_test

fraud_mask = np.array(y_explain) == 1
legit_mask = ~fraud_mask

top_feature = feature_names[np.abs(shap_values).mean(axis=0).argmax()]
top_idx = feature_names.index(top_feature)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(shap_values[legit_mask, top_idx], bins=50, alpha=0.6,
        color='#3498db', label='Legitimate', density=True)
ax.hist(shap_values[fraud_mask, top_idx], bins=50, alpha=0.6,
        color='#c0392b', label='Fraud', density=True)
ax.set_xlabel(f'SHAP Value for {top_feature}', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'SHAP Value Distribution — {top_feature}', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_fraud_vs_legit_distribution.png', dpi=300)
plt.show()
print('Saved: shap_fraud_vs_legit_distribution.png')